In [ ]:
# Colab: mount Google Drive so you can load/save the book and indices
from google.colab import drive
drive.mount('/content/drive')
# Example path to keep your files:
# /content/drive/MyDrive/book_embeddings/

In [ ]:
!pip install boto3


In [ ]:
!pip install -q pdfplumber ebooklib sentence-transformers faiss-cpu langchain chromadb tiktoken
!pip install -q openai
!pip install --upgrade openai
!pip install groq

In [ ]:
!pip install awscli


In [ ]:
!aws configure


In [ ]:
import pdfplumber  # Library used to read and extract text from PDF files

def pdf_to_text(pdf_path):
    txt = []  # This list will store text extracted from each page

    # Open the PDF file using pdfplumber
    with pdfplumber.open(pdf_path) as pdf:

        # Loop through all pages in the PDF
        for p in pdf.pages:

            # Extract text from the current page
            # If no text is found (None), it will return an empty string ""
            page_text = p.extract_text() or ""

            # Add the extracted page text to the list
            txt.append(page_text)

    # Join all page texts into one single string (separated by new lines)
    return "\n".join(txt)

# Path of your PDF file (Google Drive path in Colab)
import boto3

# -------- S3 CONFIG --------
S3_BUCKET = "ml-book-rag-data"
S3_KEY = "books/Hands-On_Machine_Learning_with_Scikit-Learn-Keras-and-TensorFlow (1).pdf"


s3 = boto3.client("s3")

# -------- DOWNLOAD PDF FROM S3 --------
local_pdf_path = "/content/mybook.pdf"

s3.download_file(
    Bucket=S3_BUCKET,
    Key=S3_KEY,
    Filename=local_pdf_path
)

# -------- Extract text --------
raw_text = pdf_to_text(local_pdf_path)



In [ ]:
import re

def clean_text(text):
    # 1. Fix broken lines (sentences split across lines)
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # 2. Remove multiple blank lines
    text = re.sub(r'\n{2,}', '\n\n', text)

    # 3. Remove standalone page numbers
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)

    # 4. Remove figure references
    text = re.sub(r'Figure\s+\d+[-–]\d+.*', '', text)
    text = re.sub(r'See Figure\s+\d+[-–]\d+', '', text)

    # 5. Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)

    # 6. Trim
    text = text.strip()

    return text

In [ ]:
text = clean_text(raw_text)
print("Characters:", len(text))
print("Sample:\n", text[1000:2000])

In [ ]:
def chunk_text(text, max_chars=2000, overlap=200):
    paragraphs = text.split("\n\n")
    chunks = []
    cur = ""
    for p in paragraphs:
        if len(cur) + len(p) + 2 <= max_chars:
            cur += ("\n\n" + p) if cur else p
        else:
            if cur:
                chunks.append(cur.strip())
            # start new chunk (if paragraph > max_chars, split paragraph)
            if len(p) > max_chars:
                for i in range(0, len(p), max_chars - overlap):
                    chunks.append(p[i:i + max_chars].strip())
                cur = ""
            else:
                cur = p
    if cur:
        chunks.append(cur.strip())
    return chunks

In [ ]:
chunks = chunk_text(text, max_chars=1500, overlap=200)
print("Chunks:", len(chunks))
print("Example chunk length (chars):", [len(c) for c in chunks[:5]])

Chunks: 163
Example chunk length (chars): [199, 1500, 1499, 1499, 1500]


In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeds = model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)
print("Embeddings shape:", embeds.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Embeddings shape: (163, 384)


In [ ]:
import faiss
import json

# ✅ Create FAISS index
dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeds)

# ✅ Save FAISS index
import boto3

# -------- S3 CONFIG --------
S3_BUCKET = "ml-book-rag-data"
FAISS_S3_KEY = "faiss/book_index.faiss"
CHUNKS_S3_KEY = "faiss/book_chunks.json"

s3 = boto3.client("s3")

# -------- LOCAL TEMP PATHS --------
local_faiss_path = "/content/book_index.faiss"
local_chunks_path = "/content/book_chunks.json"

# -------- SAVE LOCALLY --------
faiss.write_index(index, local_faiss_path)

with open(local_chunks_path, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

# -------- UPLOAD TO S3 --------
s3.upload_file(local_faiss_path, S3_BUCKET, FAISS_S3_KEY)
s3.upload_file(local_chunks_path, S3_BUCKET, CHUNKS_S3_KEY)

print("✅ FAISS index + JSON chunks uploaded to S3 successfully!")


✅ FAISS index + JSON chunks uploaded to S3 successfully!
✅ FAISS index + JSON chunks saved successfully!


# Load the saved Faiss and json files

In [ ]:
import boto3
import faiss, json

# -------- S3 CONFIG --------
S3_BUCKET = "ml-book-rag-data"
FAISS_S3_KEY = "faiss/book_index.faiss"
CHUNKS_S3_KEY = "faiss/book_chunks.json"

s3 = boto3.client("s3")

# -------- LOCAL TEMP PATHS --------
local_faiss_path = "/content/book_index.faiss"
local_chunks_path = "/content/book_chunks.json"

# -------- DOWNLOAD FROM S3 --------
s3.download_file(S3_BUCKET, FAISS_S3_KEY, local_faiss_path)
s3.download_file(S3_BUCKET, CHUNKS_S3_KEY, local_chunks_path)

# -------- LOAD --------
index = faiss.read_index(local_faiss_path)
chunks = json.load(open(local_chunks_path, "r", encoding="utf-8"))

print("✅ FAISS index + chunks loaded from S3 successfully!")


✅ FAISS index + chunks loaded from S3 successfully!


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
import numpy as np

def retrieve_faiss(query, index, chunks, model, k=5, use_openai=False):
    # 1. Create query embedding
    if use_openai:
        resp = openai.Embedding.create(
            model="text-embedding-3-large",
            input=[query]
        )
        q_emb = np.array(resp['data'][0]['embedding']).astype('float32')
    else:
        q_emb = model.encode([query])[0].astype('float32')

    # 2. Search in FAISS
    q_emb = np.array([q_emb])  # FAISS requires 2D array
    D, I = index.search(q_emb, k)

    # 3. Collect results
    results = []
    for dist, idx in zip(D[0], I[0]):
        results.append({
            "chunk_id": int(idx),
            "score": float(dist),
            "text": chunks[idx]
        })

    return results

In [ ]:

query = "What is machine learning"
results = retrieve_faiss(query, index, chunks, model, k=5, use_openai=False)

for r in results:
    print("Score:", r['score'])
    print(r['text'][:200])
    print("-----\n")


Score: 0.7127223610877991
CHAPTER 1 The Machine Learning Landscape When most people hear “Machine Learning,” they picture a robot: a dependable but‐ ler or a deadly Terminator, depending on who you ask. But Machine Learning is
-----

Score: 0.7605686783790588
PART I The Fundamentals of Machine Learning
-----

Score: 0.914203405380249
t the map and learn about the main regions and the most notable landmarks: supervised versus unsupervised learning, online versus batch learning, instance- based versus model-based learning. Then we w
-----

Score: 1.0023841857910156
Preface The Machine Learning Tsunami In 2006, Geoffrey Hinton et al. published a paper1 showing how to train a deep neural network capable of recognizing handwritten digits with state-of-the-art preci
-----

Score: 1.0126469135284424
iar with Python’s scientific libraries, the provided Jupyter note‐ books include a few tutorials. There is also a quick math tutorial for linear algebra. Roadmap This book is organized in two part

# RAG

In [ ]:
def build_context(results, max_chars=4000):
    context = ""
    for r in results:
        if len(context) + len(r["text"]) <= max_chars:
            context += r["text"] + "\n\n"
        else:
            break
    return context.strip()

context = build_context(results)
print(context[:1000])

CHAPTER 1 The Machine Learning Landscape When most people hear “Machine Learning,” they picture a robot: a dependable but‐ ler or a deadly Terminator, depending on who you ask. But Machine Learning is not just a futuristic fantasy; it’s already here. In fact, it has been around for decades in some specialized applications, such as Optical Character Recognition (OCR). But the first ML application that really became mainstream, improving the lives of hundreds of millions of people, took over the world back in the 1990s: the spam filter. It’s not exactly a self-aware Skynet, but it does technically qualify as Machine Learning (it has actually learned so well that you seldom need to flag an email as spam anymore). It was followed by hundreds of ML applications that now quietly power hundreds of products and features that you use regularly, from better recommendations to voice search. Where does Machine Learning start and where does it end? What exactly does it mean for a machine to learn s

In [ ]:
def build_memory_context(chat_history, last_n=6):
    # take last N messages (like 3 user+assistant pairs)
    recent = chat_history[-last_n:]
    memory_text = ""
    for msg in recent:
        memory_text += f"{msg['role'].upper()}: {msg['content']}\n"
    return memory_text.strip()


In [ ]:
from groq import Groq

client = Groq(api_key="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")

completion = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "Hello"}]
)

print(completion.choices[0].message.content)


How can I help you today?


In [ ]:
def build_prompt(context, query, memory):
    return f"""
You are an AI assistant.

You must answer using:
1) Memory (previous conversation)
2) Book Context (retrieved chunks)

If answer is not in Book Context, say: "Not found in the book".

Conversation Memory:
{memory}

Book Context:
{context}

User Question:
{query}

Answer:
"""


In [ ]:
chat_history = []

def ask_bot(query, index, chunks, model, k=5):
    global chat_history

    # 1) Retrieve book context
    results = retrieve_faiss(query, index, chunks, model, k=k)
    context = build_context(results)

    # 2) Build memory
    memory = build_memory_context(chat_history, last_n=6)

    # 3) Prompt
    prompt = build_prompt(context, query, memory)

    # 4) Call Groq
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    answer = completion.choices[0].message.content

    # 5) Save memory
    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": answer})

    return answer


In [ ]:
print(ask_bot("Explain PCA", index, chunks, model))
print("+++++++++++++++++++++++++++++++++++++++++++++++++++")
print(ask_bot("explain in short and give with page number also", index, chunks, model))


PCA (Principal Component Analysis) is a dimensionality reduction technique that is explained in the book. According to the book, PCA is used for preserving the variance (page 219) and its main goal is to find the principal components (page 219). The process involves projecting down to d dimensions (page 221) and it can be used for compression (page 224). The book also explains how to use Scikit-Learn for PCA (page 222) and how to choose the right number of dimensions using the explained variance ratio (page 222). Additionally, there are variations of PCA, such as Kernel PCA (page 226-230), Randomized PCA (page 225), and Incremental PCA (page 225).
+++++++++++++++++++++++++++++++++++++++++++++++++++
PCA (Principal Component Analysis) is a dimensionality reduction technique used for preserving variance (page 219). Its main goal is to find principal components (page 219) and can be used for compression (page 224). Notable variations include Kernel PCA (page 226-230), Randomized PCA (page 

In [ ]:
query = "explain pca"

results = retrieve_faiss(query, index, chunks, model, k=5)
context = "\n\n".join([r["text"] for r in results])

prompt = build_prompt(context, query,chat_history)


In [ ]:
completion = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2
)

answer = completion.choices[0].message.content
print("MODEL ANSWER:\n")
print(answer)


MODEL ANSWER:

PCA (Principal Component Analysis) is a dimensionality reduction technique used for preserving variance (page 219). Its main goal is to find principal components (page 219) and can be used for compression (page 224). Notable variations include Kernel PCA (page 226-230), Randomized PCA (page 225), and Incremental PCA (page 225).


In [ ]:
!pip install streamlit pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 98.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import pdfplumber
import re
import os
import json
import numpy as np
from datetime import datetime

import faiss
from sentence_transformers import SentenceTransformer
from groq import Groq

import boto3
from botocore.exceptions import ClientError

# ----------------------------
# ✅ S3 CONFIG (ADD)
# ----------------------------
S3_BUCKET = "ml-book-rag-data"
S3_PREFIX = "book-chatbot"

s3 = boto3.client("s3")



# ----------------------------
# ✅ CONFIG
# ----------------------------
st.set_page_config(page_title="📘 Book Chatbot (RAG + Memory)", page_icon="📘", layout="wide")

OUTPUT_FOLDER = "processed_output"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

FAISS_INDEX_PATH = os.path.join(OUTPUT_FOLDER, "book_index.faiss")
CHUNKS_PATH = os.path.join(OUTPUT_FOLDER, "book_chunks.json")


# ----------------------------
# ✅ HELPERS
# ----------------------------

# ----------------------------
# ✅ S3 HELPERS (ADD)
# ----------------------------
def upload_to_s3(local_path, s3_key):
    s3.upload_file(local_path, S3_BUCKET, s3_key)


def download_from_s3(local_path, s3_key):
    s3.download_file(S3_BUCKET, s3_key, local_path)


def s3_exists(s3_key):
    try:
        s3.head_object(Bucket=S3_BUCKET, Key=s3_key)
        return True
    except ClientError:
        return False
def get_book_name(filename):
    return os.path.splitext(filename)[0]

def pdf_to_text(pdf_file):
    txt = []
    with pdfplumber.open(pdf_file) as pdf:
        for p in pdf.pages:
            page_text = p.extract_text() or ""
            txt.append(page_text)
    return "\n".join(txt)


def clean_text(text):
    # Fix broken lines
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # Remove multiple blank lines
    text = re.sub(r"\n{2,}", "\n\n", text)

    # Remove standalone page numbers
    text = re.sub(r"^\s*\d+\s*$", "", text, flags=re.MULTILINE)

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()


def chunk_text(text, chunk_size=1200, overlap=200):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end].strip())
        start = end - overlap
        if start < 0:
            start = 0
    return [c for c in chunks if len(c) > 30]


def build_context(results, max_chars=4000):
    context = ""
    for r in results:
        if len(context) + len(r["text"]) <= max_chars:
            context += r["text"] + "\n\n"
        else:
            break
    return context.strip()


def build_memory_context(chat_history, last_n=6):
    recent = chat_history[-last_n:]
    memory_text = ""
    for msg in recent:
        role = msg["role"].upper()
        memory_text += f"{role}: {msg['content']}\n"
    return memory_text.strip()


def build_prompt(context, query, memory):
    return f"""
You are an AI assistant answering questions from a book.

INSTRUCTIONS (follow strictly):
- Explain the concept clearly and directly, as if teaching a student.
- Use the Book Context as the source of truth.
- Do NOT mention sessions, chapters, or that the book "covers" a topic.
- Do NOT give references or meta commentary.
- If the answer is not present in the Book Context, say exactly:
  "Not found in the book."

Conversation Memory:
{memory}

Book Context:
{context}

User Question:
{query}

Answer (explain clearly, in your own words, grounded in the context):
""".strip()



def create_faiss_index(embeds: np.ndarray):
    dim = embeds.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeds.astype("float32"))
    return index


def save_index_and_chunks(index, chunks, book_name):
    faiss.write_index(index, FAISS_INDEX_PATH)

    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    upload_to_s3(
        FAISS_INDEX_PATH,
        f"{S3_PREFIX}/faiss/{book_name}/index.faiss"
    )
    upload_to_s3(
        CHUNKS_PATH,
        f"{S3_PREFIX}/faiss/{book_name}/chunks.json"
    )



def load_index_and_chunks(book_name):
    faiss_key = f"{S3_PREFIX}/faiss/{book_name}/index.faiss"
    chunks_key = f"{S3_PREFIX}/faiss/{book_name}/chunks.json"

    if s3_exists(faiss_key) and s3_exists(chunks_key):
        download_from_s3(FAISS_INDEX_PATH, faiss_key)
        download_from_s3(CHUNKS_PATH, chunks_key)

        index = faiss.read_index(FAISS_INDEX_PATH)
        chunks = json.load(open(CHUNKS_PATH, "r", encoding="utf-8"))
        return index, chunks

    return None, None




def retrieve_faiss(query, index, chunks, embed_model, k=5):
    q_emb = embed_model.encode([query], convert_to_numpy=True)[0].astype("float32")
    q_emb = np.array([q_emb])

    D, I = index.search(q_emb, k)

    results = []
    for dist, idx in zip(D[0], I[0]):
        if idx < len(chunks):
            results.append({
                "chunk_id": int(idx),
                "score": float(dist),
                "text": chunks[idx]
            })
    return results


# ----------------------------
# ✅ SESSION STATE INIT
# ----------------------------
if "embed_model" not in st.session_state:
    st.session_state.embed_model = SentenceTransformer("all-MiniLM-L6-v2")

if "index" not in st.session_state:
    st.session_state.index = None

if "chunks" not in st.session_state:
    st.session_state.chunks = []

if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

if "raw_text" not in st.session_state:
    st.session_state.raw_text = ""

if "cleaned_text" not in st.session_state:
    st.session_state.cleaned_text = ""


# ----------------------------
# ✅ UI HEADER
# ----------------------------
st.title("📘 Book Chatbot (RAG + Conversational Memory)")
st.write("Upload a PDF → process into chunks → create FAISS index → chat with your book ✅")


# ----------------------------
# ✅ SIDEBAR SETTINGS
# ----------------------------
st.sidebar.header("⚙️ Settings")

# ----------------------------
# 📚 BOOK SELECTION (Step 5)
# ----------------------------
st.sidebar.subheader("📚 Select Book")

def list_books_from_s3():
    resp = s3.list_objects_v2(
        Bucket=S3_BUCKET,
        Prefix=f"{S3_PREFIX}/faiss/",
        Delimiter="/"
    )

    books = []
    for p in resp.get("CommonPrefixes", []):
        books.append(p["Prefix"].split("/")[-2])

    return books


available_books = list_books_from_s3()

if available_books:
    selected_book = st.sidebar.selectbox(
        "Choose a book",
        available_books
    )
    st.session_state.selected_book = selected_book
else:
    st.sidebar.info("No books indexed yet.")
    st.session_state.selected_book = None


k = st.sidebar.slider("Top-K Retrieved Chunks", min_value=1, max_value=10, value=5)
max_context_chars = st.sidebar.slider("Max Context Size (chars)", min_value=1000, max_value=12000, value=4000, step=500)

st.sidebar.subheader("🔑 Groq API Key")
groq_api_key = st.sidebar.text_input("Enter Groq API Key", type="password")

if groq_api_key:
    st.session_state.groq_client = Groq(api_key=groq_api_key)


# ----------------------------
# ✅ LOAD OLD INDEX IF EXISTS
# ----------------------------
# ----------------------------
# ✅ STEP 6: LOAD FAISS FOR SELECTED BOOK
# ----------------------------
if st.session_state.selected_book:
    if (
        st.session_state.index is None
        or st.session_state.get("loaded_book") != st.session_state.selected_book
    ):
        index, chunks = load_index_and_chunks(st.session_state.selected_book)

        if index is None:
            st.warning("No FAISS index found for selected book.")
        else:
            st.session_state.index = index
            st.session_state.chunks = chunks
            st.session_state.loaded_book = st.session_state.selected_book
            st.success(
                f"Loaded book: {st.session_state.selected_book} "
                f"(chunks: {len(chunks)})"
            )


    if st.button("🧹 Clear Chat Memory"):
        st.session_state.chat_history = []
        st.success("Chat memory cleared ✅")

    if st.button("🗑️ Delete Saved Index Files"):
        if os.path.exists(FAISS_INDEX_PATH):
            os.remove(FAISS_INDEX_PATH)
        if os.path.exists(CHUNKS_PATH):
            os.remove(CHUNKS_PATH)
        st.session_state.index = None
        st.session_state.chunks = []
        st.warning("Deleted saved index + chunk files ✅")


# ----------------------------
# ✅ PDF UPLOAD + PROCESS
# ----------------------------
st.subheader("📂 Step 1: Upload PDF and Process")

uploaded_pdf = st.file_uploader("Upload PDF Book", type=["pdf"])
if uploaded_pdf is not None:
    book_name = os.path.splitext(uploaded_pdf.name)[0]
    st.session_state.book_name = book_name

# Chunking Settings
st.subheader("✂️ Step 2: Chunking Settings")
col1, col2 = st.columns(2)
with col1:
    chunk_size = st.number_input("Chunk Size", min_value=200, max_value=5000, value=1200, step=100)
with col2:
    overlap = st.number_input("Chunk Overlap", min_value=0, max_value=1000, value=200, step=50)

if uploaded_pdf is not None:
    st.success("✅ PDF Uploaded Successfully!")

    if st.button("🚀 Extract + Clean + Chunk"):
        with st.spinner("⏳ Extracting text..."):
            raw_text = pdf_to_text(uploaded_pdf)
            st.session_state.raw_text = raw_text

        with st.spinner("🧹 Cleaning text..."):
            cleaned = clean_text(raw_text)
            st.session_state.cleaned_text = cleaned

        with st.spinner("✂️ Chunking..."):
            chunks = chunk_text(cleaned, chunk_size=chunk_size, overlap=overlap)
            st.session_state.chunks = chunks

        st.success(f"✅ Done! Total Chunks: {len(chunks)}")

        st.subheader("✅ Cleaned Text Preview")
        st.text_area("Cleaned Text", cleaned[:3000], height=250)

        st.subheader("📦 Chunk Preview (First 3)")
        for i in range(min(3, len(chunks))):
            st.markdown(f"### Chunk {i+1}")
            st.write(chunks[i])

        st.download_button(
            "⬇️ Download Cleaned Text (.txt)",
            data=cleaned,
            file_name="cleaned_text.txt",
            mime="text/plain"
        )

        st.download_button(
            "⬇️ Download Chunks (.txt)",
            data="\n\n".join(chunks),
            file_name="chunks.txt",
            mime="text/plain"
        )


# ----------------------------
# ✅ CREATE EMBEDDINGS + FAISS INDEX
# ----------------------------
st.subheader("🧠 Step 3: Create Embeddings + FAISS Index")

if st.session_state.chunks:
    if st.button("⚡ Create FAISS Index Now"):
        with st.spinner("🔄 Creating embeddings... (one-time operation)"):
            embeds = st.session_state.embed_model.encode(
                st.session_state.chunks,
                batch_size=32,
                show_progress_bar=True,
                convert_to_numpy=True
            ).astype("float32")

        with st.spinner("📌 Building FAISS index..."):
            index = create_faiss_index(embeds)
            st.session_state.index = index

        save_index_and_chunks(
            index,
            st.session_state.chunks,
            st.session_state.book_name
        )

        st.success("✅ FAISS index created and saved to S3!")


# ----------------------------
# ✅ CHATBOT SECTION
# ----------------------------
st.subheader("💬 Step 4: Chat With Your Book (Memory Enabled)")

if st.session_state.index is None or not st.session_state.chunks:
    st.info("⚠️ Please upload/process a PDF and create/load FAISS index first.")
else:
    # Show chat history
    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    user_query = st.chat_input("Ask something from the book...")

    if user_query:
        # Show user msg
        st.session_state.chat_history.append({"role": "user", "content": user_query})
        with st.chat_message("user"):
            st.markdown(user_query)

        # Validate key
        if "groq_client" not in st.session_state:
            with st.chat_message("assistant"):
                st.error("❌ Please enter your Groq API key in sidebar.")
        else:
            with st.chat_message("assistant"):
                with st.spinner("🔎 Searching book + generating answer..."):
                    # 1) Retrieve from FAISS
                    results = retrieve_faiss(
                        user_query,
                        st.session_state.index,
                        st.session_state.chunks,
                        st.session_state.embed_model,
                        k=k
                    )

                    # 2) Build context
                    context = build_context(results, max_chars=max_context_chars)

                    # 3) Build memory
                    memory = build_memory_context(st.session_state.chat_history, last_n=6)

                    # 4) Prompt
                    prompt = build_prompt(context, user_query, memory)

                    # 5) Call Groq
                    completion = st.session_state.groq_client.chat.completions.create(
                        model="llama-3.1-8b-instant",
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.2
                    )

                    answer = completion.choices[0].message.content

                st.markdown(answer)

            # Save assistant response to memory
            st.session_state.chat_history.append({"role": "assistant", "content": answer})

            # Optional debug: show retrieved chunks
            with st.expander("📌 Debug: Retrieved Chunks"):
                for r in results:
                    st.markdown(f"**Chunk ID:** {r['chunk_id']} | **Score:** {r['score']:.4f}")
                    st.write(r["text"][:600])
                    st.markdown("---")


Overwriting app.py


In [ ]:
!streamlit run app.py &> /content/logs.txt &
!pip install pyngrok


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("38ZtTW6OU1h9Dks3NGJWI4CetRz_7L7smeu9eE6SBNj4n4TPM")
public_url = ngrok.connect(8501)
print(public_url)



NgrokTunnel: "https://nickole-winish-sympetaly.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 117544 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.1.2) over (2026.1.2) ...
Setting up cloudflared (2026.1.2) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!streamlit run app.py &> /content/logs.txt &

In [ ]:
!cloudflared tunnel --url http://localhost:8501


2026-02-03T09:15:00Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-03T09:15:00Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-03T09:15:09Z INF +--------------------------------------------------------------------------------------------+
2026-02-03T09:15:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-02-03T09:15:09Z INF |  https://example-attempt-dose-soon.trycloudflare.com  